# Lektion 07: LangChain – vom Handbau zum Fließband



| | |
|---|---|
| **Zeit** | ~45 Minuten |
| **Schwierigkeit** | Leicht |
| **Voraussetzungen** | Lektion 05–06 (du hast Prompt-Bau & Modell-Call von Hand gebaut!) |
| **Du baust** | Deine erste Chain – das Fließband, das deinen Build-1-Handarbeit ersetzt |

## 🎯 Nach dieser Lektion kannst du:

- erklären, **welche** Handarbeit aus Build 1 LangChain ersetzt

- Nachrichten als LangChain-Objekte (`SystemMessage`, `HumanMessage`) nutzen

- eine **Chain** mit `|` verketten: Prompt → Modell → Parser

- Prompt-Templates mit Platzhaltern füllen

- den Validierungsfehler bei vergessenen Platzhaltern als Feature erklären

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧠 Konzept: Was ersetzt LangChain hier konkret?

**WARUM:** Erinner dich an Build 1: `baue_prompt` war String-Kleberei, der Client-Call roh, die Antwort musertest du aus `choices[0].message.content` wühlen – und das **jedes Mal wieder**.

**WAS:** LangChain liefert standardisierte **Bausteine** mit einer gemeinsamen Schnittstelle (Runnable): alles hat `.invoke()` und lässt sich mit `|` wie ein Fließband verketten.

| Build 1 (dein Handbau) | Build 2 (LangChain-Baustein) |
|---|---|
| `baue_prompt` (String-Konkatenation) | `ChatPromptTemplate` |
| `client.chat.complete(...)` | Chat-Modell-Objekt |
| `antwort.choices[0].message.content` | `StrOutputParser()` |

**Metapher Chain = Fließband:** Ein Arbeiter schneidet Bleche (Template füllt die Werte), der nächste lackiert (Modell antwortet), der letzte prüft und verpackt (Parser macht einen sauberen String). Übergabepunkt ist das Band selbst – die Chain.

## 🔎 Beispiel: Nachrichten als Objekte

LangChain spricht nicht mehr mit rohen Dictionaries, sondern mit **Message-Objekten:

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage
from lernakademie.llm_client import get_chat_model

modell = get_chat_model(temperature=0.2)

antwort = modell.invoke([
    SystemMessage(content="Du bist ein Python-Tutor fuer absolute Anfaenger."),
    HumanMessage(content="Erklaere in einem Satz, was eine Variable ist."),
])
print("Antwort-Objekt:", antwort)
print("\nNur der Text (.content):")
print(antwort.content)

Antwort-Objekt: content='[OFFLINE] Simulierte Antwort (kein echtes Modell, keine Kosten). Ein echtes Modell wuerde jetzt auf folgenden Text reagieren: „Erklaere in einem Satz, was eine Variable ist.“' additional_kwargs={} response_metadata={} id='lc_run--01a1a3a0-e469-7756-a782-2fb4a6130254-0' tool_calls=[] invalid_tool_calls=[]

Nur der Text (.content):
[OFFLINE] Simulierte Antwort (kein echtes Modell, keine Kosten). Ein echtes Modell wuerde jetzt auf folgenden Text reagieren: „Erklaere in einem Satz, was eine Variable ist.“


### 🔎 Was sehen wir hier?
Das Modell-Objekt kennt keine Dictionaries mehr – nur noch typisierte Nachrichten. Die Antwort ist ein `AIMessage`; der Text steckt in `.content`. Noch ist es Handarbeit – jetzt kommt das Fließband:

## 🧠 Konzept: Prompt-Template & Chain

**WAS:** Ein `ChatPromptTemplate` ist eine **Vorlage mit Platzhaltern** (`{stil}`, `{text}`). Statt Strings zu kleben, füllst du beim Aufruf ein Dictionary aus.

**WIE (das berühmte Einzeiler-Muster):**

`chain = prompt | modell | parser`  ← das komplette Fließband in einer Zeile.

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Du fasst Texte praegnant und korrekt zusammen."),
    ("human", "Stil: {stil}. Zusammenfassen: {text}"),
])
chain = prompt | modell | StrOutputParser()

BEISPIELTEXT = ("Python ist eine Programmiersprache, die 1991 veroeffentlicht wurde. "
                "Sie ist bekannt fuer ihre gut lesbare Syntax und wird von der Webentwicklung "
                "bis zur Datenanalyse eingesetzt.")

for stil in ("kurz", "als Bullet Points"):
    ergebnis = chain.invoke({"stil": stil, "text": BEISPIELTEXT})
    print(f"--- Stil: {stil} ---")
    print(ergebnis[:200], "\n")

--- Stil: kurz ---
[OFFLINE] Simulierte Antwort (kein echtes Modell, keine Kosten). Ein echtes Modell wuerde jetzt auf folgenden Text reagieren: „Stil: kurz. Zusammenfassen: Python ist eine Programmiersprache, die 1991  

--- Stil: als Bullet Points ---
[OFFLINE] Simulierte Antwort (kein echtes Modell, keine Kosten). Ein echtes Modell wuerde jetzt auf folgenden Text reagieren: „Stil: als Bullet Points. Zusammenfassen: Python ist eine Programmiersprac 



### 🔎 Was sehen wir hier?
1. **Dieselbe Chain**, zwei Stile – nur die **Eingabe** (das Dictionary) ändert sich. Das ist Wiederverwendbarkeit.
2. Das Ergebnis ist ein **String** (Parser!) – kein `AIMessage`-Objekt mehr.
3. Die Chain ist ein Runnable: `.invoke(dict)` links rein, fertiges Ergebnis rechts raus.

In [4]:
# Was passiert bei einem vergessenen Platzhalter? Fuehre aus – der Fehler ist ein FEATURE:
try:
    chain.invoke({"stil": "kurz"})   # {text} fehlt!
except Exception as fehler:
    print("Fehlerart:", type(fehler).__name__)
    print("Meldung:", str(fehler)[:150])

Fehlerart: KeyError
Meldung: "Input to ChatPromptTemplate is missing variables {'text'}.  Expected: ['stil', 'text'] Received: ['stil']\nNote: if you intended {text} to be part of


** Warum das gut ist:** LangChain **validiert** deine Eingabe gegen das Template – vergessene Werte Crashen nicht mittendrin, sondern gleich am Eingang mit klarer Meldung. In Build 1 hättest du das selbst programmieren müssen.

## ✏️ Deine Aufgabe: Deine eigene Übersetzer-Chain

Du baust selbst ein zweites Fließband – mit eigenen Platzhaltern.

**Anforderungen:**

- Erstelle `uebersetzer_prompt` mit den Platzhaltern `{text}` und `{zielsprache}` (System-Rolle: „Du übersetzt natürlich und idiomatisch.“).

- Verkette eine Chain `uebersetzer = uebersetzer_prompt | modell | StrOutputParser()`.

- Übersetze den Beispielsatz nach Englisch und nach Spanisch (zwei `invoke`-Aufrufe, dieselbe Chain).

**Schreibe deinen Code in die folgende Zelle:**

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

uebersetzer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Du uebersetzt natuerlich und idiomatisch."),
    ("human", "Uebersetze nach {zielsprache}: {text}"),
])
uebersetzer = uebersetzer_prompt | modell | StrOutputParser()

print(uebersetzer.invoke({"text": "Guten Morgen", "zielsprache": "Englisch"})[:120])
print(uebersetzer.invoke({"text": "Guten Morgen", "zielsprache": "Spanisch"})[:120])

[OFFLINE] Simulierte Antwort (kein echtes Modell, keine Kosten). Ein echtes Modell wuerde jetzt auf folgenden Text reagi
[OFFLINE] Simulierte Antwort (kein echtes Modell, keine Kosten). Ein echtes Modell wuerde jetzt auf folgenden Text reagi


**💡 Warum diese Loesung funktioniert:**

Template, Modell und Parser sind unverändert – nur die Vorlage ist neu. Genau das ist der Punkt: Bausteine austauschen statt Code neu schreiben. Mit echtem Modell übersetzt die Chain wirklich; im Offline-Modus siehst du die simulierte Antwort – die Mechanik ist identisch.

### ✅ Richtig, wenn:

- [ ] Die Chain liefert Strings (keine `AIMessage`-Objekte).

- [ ] Beide Sprachen funktionieren über **dieselbe** Chain.

- [ ] Ein vergessener Platzhalter im `invoke` erzeugt einen Validierungsfehler (Feature!).

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [6]:
ergebnis = uebersetzer.invoke({"text": "Guten Morgen", "zielsprache": "Englisch"})
assert isinstance(ergebnis, str) and len(ergebnis) > 0
try:
    uebersetzer.invoke({"text": "Guten Morgen"})   # zielsprache fehlt
    raise AssertionError("Hier haette ein Validierungsfehler kommen muessen!")
except AssertionError:
    raise
except Exception:
    print("Validierung greift ✅")
print("Alle Prüfungen bestanden! ✅")

Validierung greift ✅
Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Template: `ChatPromptTemplate.from_messages([("system", "..."), ("human", "Uebersetze nach {zielsprache}: {text}")])`
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Der Parser importiert aus `langchain_core.output_parsers`.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Zähle: Wie viele Runnables stecken in `prompt | modell | parser`? (Tipp: 3!)

2. Was hat LangChain gegenüber Build 1 **einfacher** gemacht – und was wurde **komplizierter** (neue Begriffe, Abstraktion)?

## ➡️ Weiter

**Weiter mit Lektion 08:** `notebooks/04_langchain/08_embeddings_vektor_rag.ipynb` – der große Qualitätssprung: **Embeddings** ersetzen deine Wort-Zettel-Suche durch Bedeutung-Suche.